# 实验5：使用 PyAsc 编写简单算子

## 学习目标

1. 使用 PyAsc 编写完整的 NPU 向量加法 Kernel
2. 理解 GlobalAddress / GlobalTensor / LocalTensor 的内存建模
3. 掌握 data_copy、set_flag、wait_flag 的事件同步机制
4. 建立 Python -> ASC-IR -> Ascend C -> NPU Kernel 的编译链路认知

## 环境准备

CANNLab 已预装 CANN 工具链和 Python 3.12。安装 PyAsc 和依赖：

In [ ]:
# 如未克隆 PyAsc 源码，先克隆
cd ~
!git clone https://gitcode.com/cann/pyasc.git 2>/dev/null || echo "已存在，跳过"
cd pyasc

# 安装依赖
!pip install -r requirements-build.txt -q
!pip install -r requirements-runtime.txt -q
!pip install pyasc torch pytest lit -q
!echo "环境就绪！" 

## 1. 分析 Add 算子规格

本实验使用 `examples/01_add/add.py`。Add 算子的数学表达式为 **z = x + y**。

**规格参数：**
- 输入/输出：float32 Tensor，shape (8 x 2048,)
- 并行策略：8 核并行，每核 2048 个元素
- 流水策略：TILE_NUM=8, BUFFER_NUM=2（双缓冲）
- tile_length = block_length // TILE_NUM // BUFFER_NUM

先看样例目录结构：

In [ ]:
ls -la examples/01_add/

## 2. 阅读核函数源码

In [ ]:
!cat examples/01_add/add.py

**源码要点分析：**

**1) 类型体系：**
- `asc.GlobalAddress`：NPU 全局内存地址（传入参数）
- `asc.GlobalTensor()`：全局内存张量，通过 `set_global_buffer` 绑定地址
- `asc.LocalTensor(dtype, position, offset, length)`：片上 Local Memory 张量

**2) 核函数签名：**
```python
@asc.jit
def vadd_kernel(x: asc.GlobalAddress, y: asc.GlobalAddress, z: asc.GlobalAddress, block_length: int):
```
`@asc.jit` 标记此函数为 JIT 编译入口。参数类型必须是 `asc.GlobalAddress`。

**3) 内存绑定：**
```python
x_gm = asc.GlobalTensor()
x_gm.set_global_buffer(x + offset, block_length)
```
每个核只访问自己负责的数据块（offset = block_idx * block_length）。

**4) 双缓冲 tile 切分：**
```python
tile_length = block_length // TILE_NUM // BUFFER_NUM  # = 2048 // 8 // 2 = 128
```

**5) 内核启动语法：**
```python
vadd_kernel[USE_CORE_NUM, rt.current_stream()](x, y, z, block_length)
```
用方括号 `[...]` 指定核数和流，圆括号传入参数。

**6) torch_npu 可选导入：**
```python
try:
    import torch_npu
except ModuleNotFoundError:
    pass
```
`torch_npu` 是华为维护的 PyTorch 昇腾后端，仿真模式下不需要。

## 3. 数据搬运与事件同步

NPU 内部有三个独立硬件引擎并行执行：
- **MTE2**：Global Memory -> Local Memory 搬入
- **Vector**：片上向量计算
- **MTE3**：Local Memory -> Global Memory 搬出

事件同步保证阶段顺序：

In [ ]:
# 查看同步相关的关键代码
!grep -n "data_copy\|set_flag\|wait_flag\|asc.add" examples/01_add/add.py

**同步模式：**
```
MTE2 搬入 x, y (data_copy)
  -> set_flag(MTE2_V) / wait_flag(MTE2_V)
Vector 计算 z = x + y (asc.add)
  -> set_flag(V_MTE3) / wait_flag(V_MTE3)
MTE3 搬出 z (data_copy)
  -> set_flag(MTE3_MTE2) / wait_flag(MTE3_MTE2)
```

> **思考：** 如果去掉 set_flag/wait_flag 会导致什么？

## 4. 运行 Add 算子

CANNLab 已预装 CANN simulator 库，直接运行仿真模式：

In [ ]:
cd ~/pyasc/examples/01_add
!python3 add.py -r Model -v Ascend910B1

看到 `[INFO] Sample add run success.` 即验证通过！

程序内部用 `torch.allclose(z, x + y)` 校验 NPU 结果与 CPU 基准一致。

## 5. 编译链路

```
Python (@asc.jit)  ->  ASC-IR (MLIR Op)  ->  Ascend C  ->  NPU Kernel
    前端捕获              中间表示             目标代码         可执行
```

> **思考：** PyAsc 相比直接写 Ascend C 有什么优势？为什么 Python 前端仍需遵守硬件编程模型？

## 6. 任务拓展

改写 vsub_kernel：
1. `asc.add` -> `asc.sub`
2. `torch.allclose(z, x + y)` -> `torch.allclose(z, x - y)`
3. 比较 Add/Sub 在结构上的异同

## 总结

1. PyAsc 编写 NPU 算子的完整流程
2. GlobalAddress/GlobalTensor/LocalTensor 内存建模
3. MTE2 -> Vector -> MTE3 流水线与事件同步
4. Python -> ASC-IR -> Ascend C -> NPU Kernel 端到端链路